<div style="display:flex; align-items:center; gap:16px;">
  <img src="https://i.postimg.cc/J7sf3Lq4/Group.png" alt="logo" style="height:70px;"/>
  <div>
    <h1 style="margin:0;">Sesión 4 — 2) Clases y Objetos (POO)</h1>
    <p style="margin:4px 0 0 0;">Incremental + aplicado a Data Science (sin dejar nada implícito)</p>
  </div>
</div>

---

## Objetivos
Al final de esta sesión el alumno podrá:
1) Explicar **clase vs objeto** con ejemplos.
2) Crear clases con **atributos y métodos**.
3) Usar `__init__` para inicializar estado.
4) Entender `self` y por qué existe.
5) Diferenciar **atributos de clase** vs **atributos de instancia**.
6) Aplicar **composición** y **herencia**.
7) Usar métodos mágicos (`__str__`, `__add__`, comparaciones).
8) Aplicar POO a casos típicos de Data Science: **limpieza**, **métricas**, **features**, **experimentos A/B**, **evaluación de modelos**.

### Requisitos previos (sesiones anteriores)
- `if/elif/else`, `for/while`, `break/continue`, `try/except`
- listas, diccionarios, `enumerate`
- funciones y `return`


# Parte 1 — Fundamentos de POO

## 1) Qué es una clase y qué es un objeto

- **Clase**: una plantilla (el "molde"). Describe qué datos (atributos) y qué acciones (métodos) tendrán los objetos.
- **Objeto**: una instancia real creada a partir de la clase.

Analogía:
- Clase = plano de una casa.
- Objeto = una casa construida con ese plano.


## 2) Clase mínima y por qué no es suficiente

Una clase sin atributos ni métodos se puede crear, pero no modela nada útil.

In [3]:
class Vacia:
    pass

obj = Vacia()
print(type(obj))
print(obj)

<class '__main__.Vacia'>


## 3) Atributos y métodos (primera clase útil)

- **Atributos**: guardan el estado.
- **Métodos**: cambian o usan ese estado.

Aquí hacemos una clase simple `Sensor`.

Nota: primero lo haremos con valores fijos (atributos de clase), y luego lo haremos bien con `__init__`.

In [4]:
class Sensor:
    # Atributos de CLASE (defaults)
    unidad = "C"
    activo = True

    def leer(self):
        # Método simple
        print("Leyendo sensor... unidad:", self.unidad)

s = Sensor()
print("unidad:", s.unidad)
print("activo:", s.activo)
s.leer()

unidad: C
activo: True
Leyendo sensor... unidad: C


## 4) `__init__`: el constructor (estado propio por objeto)

El método `__init__` se ejecuta automáticamente cuando creas el objeto:
```python
obj = Clase(...)
```

Se usa para inicializar **atributos de instancia**.


In [5]:
class Sensor2:
    def __init__(self, sensor_id, unidad="C"):
        self.sensor_id = sensor_id
        self.unidad = unidad
        self.activo = True
        self.lecturas = []  # lista de lecturas del sensor

    def registrar(self, valor):
        # Guardamos datos dentro del objeto
        self.lecturas.append(valor)

    def ultimo(self):
        return self.lecturas[-1] if len(self.lecturas) > 0 else None

s1 = Sensor2("S-01")
s2 = Sensor2("S-02", unidad="F")

s1.registrar(20)
s1.registrar(21)

s2.registrar(70)

print("s1 id/unidad/ultimo:", s1.sensor_id, s1.unidad, s1.ultimo())
print("s2 id/unidad/ultimo:", s2.sensor_id, s2.unidad, s2.ultimo())

s1 id/unidad/ultimo: S-01 C 21
s2 id/unidad/ultimo: S-02 F 70


## 5) `self`: explicado sin ambigüedades

`self` es "el objeto actual".

Cuando escribes:
```python
s1.registrar(21)
```
Python lo traduce a:
```python
Sensor2.registrar(s1, 21)
```

Por eso el método debe declararse con `self`.


## 6) Atributos de clase vs instancia (comparación clara)

- Atributo de clase: se define en el cuerpo de la clase.
- Atributo de instancia: se define en `__init__` con `self.`.

Si cambias el atributo de clase, afecta a instancias que NO lo hayan sobrescrito.


In [6]:
class Producto:
    impuesto = 0.18  # atributo de clase

    def __init__(self, nombre, precio):
        self.nombre = nombre
        self.precio = precio

    def precio_final(self):
        return self.precio * (1 + Producto.impuesto)

p1 = Producto("A", 100)
p2 = Producto("B", 200)

print("Impuesto (clase):", Producto.impuesto)
print("p1 final:", p1.precio_final())
print("p2 final:", p2.precio_final())

Producto.impuesto = 0.10
print("\nCambio impuesto a 10%")
print("p1 final:", p1.precio_final())
print("p2 final:", p2.precio_final())

# sobrescribir en una instancia (no recomendado para este caso)
p1.impuesto = 0.50
print("\nSi hago p1.impuesto=0.50, no afecta al método si usa Producto.impuesto")
print("p1.impuesto (instancia):", p1.impuesto)
print("Producto.impuesto (clase):", Producto.impuesto)

Impuesto (clase): 0.18
p1 final: 118.0
p2 final: 236.0

Cambio impuesto a 10%
p1 final: 110.00000000000001
p2 final: 220.00000000000003

Si hago p1.impuesto=0.50, no afecta al método si usa Producto.impuesto
p1.impuesto (instancia): 0.5
Producto.impuesto (clase): 0.1


# Parte 2 — Composición, herencia y diseño

## 7) Composición (un objeto TIENE otros objetos)

Esto es útil cuando un objeto está formado por partes.

Ejemplo DS: un `Pipeline` TIENE pasos; un `Dataset` TIENE filas.


In [11]:
class Motor:
    def __init__(self, hp):
        self.hp = hp

class Auto:
    def __init__(self, placa, motor):
        self.placa = placa
        self.motor = motor  # composición

    def __str__(self):
        return "Auto {} ({} hp)".format(self.placa, self.motor.hp)

m = Motor(150)
a = Auto("ABC-123", m)
print(a)

Auto ABC-123 (150 hp)


## 8) Herencia (un objeto ES un tipo de otro)

Ejemplo: `Rectangulo` ES una `Figura`.

Se usa cuando hay una jerarquía natural y quieres reutilizar lo común.


In [12]:
class Figura:
    def __init__(self, name):
        self.name = name

    def descripcion(self):
        return "Figura: " + self.name

class Rectangulo(Figura):
    def __init__(self, x, y, name):
        super().__init__(name)
        self.x = x
        self.y = y

    def area(self):
        return self.x * self.y

    def descripcion(self):
        # override: redefinimos el comportamiento
        return "Rectangulo {}: {}x{}".format(self.name, self.x, self.y)

r = Rectangulo(10, 5, "R1")
print(r.descripcion())
print("area:", r.area())

Rectangulo R1: 10x5
area: 50


## 9) Errores comunes: parámetros mutables en `__init__`

MAL (lista compartida):
```python
def __init__(self, items=[]):
```

BIEN:
```python
def __init__(self, items=None):
    self.items = items if items is not None else []
```


In [15]:
class MalCatalogo:
    def __init__(self, items=[]):
        self.items = items

a = MalCatalogo()
b = MalCatalogo()
a.items.append("X")
print("a.items:", a.items)
print("b.items (sorpresa):", b.items)

class BuenCatalogo:
    def __init__(self, items=None):
        self.items = items if items is not None else []

c = BuenCatalogo()
d = BuenCatalogo()
c.items.append("X")
print("c.items:", c.items)
print("d.items:", d.items)

a.items: ['X']
b.items (sorpresa): ['X']
c.items: ['X']
d.items: []


## 10) Encapsulación ligera: `property` para validar cambios

En POO, encapsulación significa controlar cómo se modifica el estado.

En Python, se usa `property` para validar sin cambiar la interfaz.

Ejemplo: un `Producto` cuyo precio no puede ser negativo.


In [3]:
class ProductoSeguro:
    def __init__(self, nombre, precio):
        self.nombre = nombre
        self.precio = precio  # pasa por el setter

    @property
    def precio(self):
        return self._precio

    @precio.setter
    def precio(self, value):
        if value < 0:
            raise ValueError("precio no puede ser negativo")
        self._precio = value

p = ProductoSeguro("A", 1)
print(p.precio)
# p.precio = -1  # descomenta para ver el error

1


# Parte 3 — Métodos mágicos (dunder)

## 11) `__str__`: cómo se muestra tu objeto
Si no lo defines, Python imprime algo como `<__main__.Clase object at ...>`.


In [22]:
class Usuario:
    def __init__(self, user_id, nombre):
        self.user_id = user_id
        self.nombre = nombre

    def __str__(self):
        return "{} ({})".format(self.nombre, self.user_id)

u = Usuario("U1", "Ana")
print(u)

Ana (U1)


## 12) `__add__` y comparaciones: ejemplo con fracciones

Esto permite que `x + y` funcione con objetos.


In [4]:
def mcd(m, n):
    # mcd clásico
    while m % n != 0:
        m, n = n, m % n
    return n

class Fraccion:
    def __init__(self, num, den):
        if den == 0:
            raise ValueError("denominador no puede ser 0")
        self.num = num
        self.den = den

    def __str__(self):
        return str(self.num) + "/" + str(self.den)

    def __add__(self, otra):
        nuevo_num = self.num * otra.den + self.den * otra.num
        nuevo_den = self.den * otra.den
        comun = mcd(abs(nuevo_num), abs(nuevo_den))
        return Fraccion(nuevo_num // comun, nuevo_den // comun)

    def __eq__(self, otra):
        return self.num * otra.den == otra.num * self.den

    def __gt__(self, otra):
        return self.num * otra.den > otra.num * self.den

x = Fraccion(1, 2)
y = Fraccion(2, 4)
z = Fraccion(3, 4)

print("x:", x)
print("y:", y)
print("x == y:", x == y)
print("x + z:", x + z)
print("z > x:", z > x)

x: 1/2
y: 2/4
x == y: True
x + z: 5/4
z > x: True


# Parte 4 — POO aplicada a Data Science

La idea es que los alumnos vean **por qué** POO sirve en DS:
- organizar limpieza/validación
- agrupar métricas
- encapsular reglas
- crear mini-herramientas reutilizables


## DS 1) `Record`: una fila con validación + conversión segura

Modelamos una "fila" de datos como objeto.

Ventaja:
- un lugar único para limpiar/validar
- reuso en muchos ejercicios


In [16]:
class Record:
    def __init__(self, row):
        self.row = row

    def to_float_safe(self, key):
        x = self.row.get(key)
        try:
            if x is None or x == "" or x == "NA":
                return None
            if type(x) == str:
                x = x.strip()
                if x == "":
                    return None
            return float(x)
        except:
            return None

    def is_valid(self):
        # Reglas de validación ejemplo: gasto y visitas deben ser válidos
        g = self.to_float_safe("gasto")
        v = self.to_float_safe("visitas")
        return (g is not None) and (v is not None)

rows = [
    {"id": "C1", "gasto": "120", "visitas": "3"},
    {"id": "C2", "gasto": "NA", "visitas": "1"},
    {"id": "C3", "gasto": "80", "visitas": None},
]

for r in rows:
    rec = Record(r)
    print(r["id"], "valid?", rec.is_valid(), "gasto:", rec.to_float_safe("gasto"), "visitas:", rec.to_float_safe("visitas"))

C1 valid? True gasto: 120.0 visitas: 3.0
C2 valid? False gasto: None visitas: 1.0
C3 valid? False gasto: 80.0 visitas: None


## DS 2) `Dataset`: métricas + estilo `fit/transform` (sin librerías)

Creamos un Dataset con:
- `fit(cols)`: calcula promedios por columna (ignorando inválidos)
- `transform(cols)`: devuelve un dataset nuevo donde reemplaza inválidos por el promedio

Esto es similar a un preprocesamiento típico.


In [17]:
class Dataset:
    def __init__(self, rows):
        self.rows = rows
        self.promedios = {}

    def _to_float_safe(self, x):
        try:
            if x is None or x == "" or x == "NA":
                return None
            if type(x) == str:
                x = x.strip()
                if x == "":
                    return None
            return float(x)
        except:
            return None

    def fit(self, cols):
        # calcula promedios por columna
        for c in cols:
            total = 0
            n = 0
            for r in self.rows:
                v = self._to_float_safe(r.get(c))
                if v is None:
                    continue
                total += v
                n += 1
            self.promedios[c] = (total / n) if n > 0 else None
        return self

    def transform(self, cols):
        # retorna NUEVAS filas (no muta el dataset original)
        nuevas = []
        for r in self.rows:
            nuevo = {}
            for k, v in r.items():
                nuevo[k] = v

            for c in cols:
                v = self._to_float_safe(nuevo.get(c))
                if v is None:
                    # imputación simple por promedio
                    nuevo[c] = self.promedios.get(c)
                else:
                    nuevo[c] = v

            nuevas.append(nuevo)

        return Dataset(nuevas)

rows = [
    {"id": "C1", "gasto": "120", "visitas": "3"},
    {"id": "C2", "gasto": "NA",  "visitas": "1"},
    {"id": "C3", "gasto": "80",  "visitas": None},
    {"id": "C4", "gasto": "200", "visitas": "5"},
]

ds = Dataset(rows)
ds.fit(["gasto", "visitas"])
print("Promedios aprendidos:", ds.promedios)

ds2 = ds.transform(["gasto", "visitas"])
print("\nDataset transformado:")
for r in ds2.rows:
    print(r)

Promedios aprendidos: {'gasto': 133.33333333333334, 'visitas': 3.0}

Dataset transformado:
{'id': 'C1', 'gasto': 120.0, 'visitas': 3.0}
{'id': 'C2', 'gasto': 133.33333333333334, 'visitas': 1.0}
{'id': 'C3', 'gasto': 80.0, 'visitas': 3.0}
{'id': 'C4', 'gasto': 200.0, 'visitas': 5.0}


## DS 3) `ExperimentAB`: conversión y ganador + resumen

POO sirve para empaquetar el cálculo y el reporte.


In [18]:
class ExperimentAB:
    def __init__(self, eventos_por_variante):
        self.eventos = eventos_por_variante

    def conversion_rate(self, variante):
        ev = self.eventos.get(variante, [])
        if len(ev) == 0:
            return None
        conv = 0
        for x in ev:
            if x == 1:
                conv += 1
        return conv / len(ev)

    def resumen(self):
        out = {}
        for v in self.eventos:
            out[v] = {
                "n": len(self.eventos[v]),
                "cr": self.conversion_rate(v)
            }
        return out

    def winner(self):
        mejor_v = None
        mejor_cr = -1
        for v in self.eventos:
            cr = self.conversion_rate(v)
            if cr is not None and cr > mejor_cr:
                mejor_cr = cr
                mejor_v = v
        return mejor_v, mejor_cr

ab = {
    "A": [0,1,0,1,0,0,1,0],
    "B": [1,1,0,1,1,0,1,0],
    "C": [0,0,0,1,0,0,0,0],
}

exp = ExperimentAB(ab)
print("Resumen:", exp.resumen())
print("Winner:", exp.winner())

Resumen: {'A': {'n': 8, 'cr': 0.375}, 'B': {'n': 8, 'cr': 0.625}, 'C': {'n': 8, 'cr': 0.125}}
Winner: ('B', 0.625)


## DS 4) `ConfusionMatrix`: evaluación de clasificación

Incluye métricas: accuracy, precision, recall.


In [ ]:
class ConfusionMatrix:
    def __init__(self, y_true, y_pred):
        self.y_true = y_true
        self.y_pred = y_pred
        self.tp = 0
        self.tn = 0
        self.fp = 0
        self.fn = 0
        self._calc()

    def _calc(self):
        for yt, yp in zip(self.y_true, self.y_pred):
            if yt == 1 and yp == 1:
                self.tp += 1
            elif yt == 0 and yp == 0:
                self.tn += 1
            elif yt == 0 and yp == 1:
                self.fp += 1
            else:
                self.fn += 1

    def accuracy(self):
        n = len(self.y_true)
        return (self.tp + self.tn) / n if n > 0 else None

    def precision(self):
        denom = self.tp + self.fp
        return self.tp / denom if denom > 0 else None

    def recall(self):
        denom = self.tp + self.fn
        return self.tp / denom if denom > 0 else None

    def __str__(self):
        return "TP:{} TN:{} FP:{} FN:{}".format(self.tp, self.tn, self.fp, self.fn)

y_true = [1,0,1,1,0,0,1,0,1,0]
y_pred = [1,0,0,1,0,1,1,0,1,0]

cm = ConfusionMatrix(y_true, y_pred)
print(cm)
print("acc:", cm.accuracy())
print("prec:", cm.precision())
print("rec:", cm.recall())

## DS 5) `MetricTracker`: métricas en streaming (update con while/for)

Caso real: llegan datos de a uno (streaming). Queremos actualizar:
- conteo
- suma
- mínimo/máximo
- promedio

Esto conecta con bucles + acumuladores + POO.


In [ ]:
class MetricTracker:
    def __init__(self):
        self.n = 0
        self.total = 0
        self.minimo = None
        self.maximo = None

    def update(self, x):
        # valida x
        if x is None:
            return
        self.n += 1
        self.total += x
        if self.minimo is None or x < self.minimo:
            self.minimo = x
        if self.maximo is None or x > self.maximo:
            self.maximo = x

    def mean(self):
        return self.total / self.n if self.n > 0 else None

    def resumen(self):
        return {"n": self.n, "min": self.minimo, "max": self.maximo, "mean": self.mean()}

tracker = MetricTracker()
stream = [10, 12, None, 9, 11]

for x in stream:
    tracker.update(x)

print(tracker.resumen())

## DS 6) `FeatureRuleEngine`: motor de reglas (features interpretables)

Caso común: crear etiquetas o features por reglas (muy usado incluso antes de modelos ML).

Haremos un motor simple que:
- recibe reglas por sector
- etiqueta cada registro
- cuenta resultados


In [ ]:
class FeatureRuleEngine:
    def __init__(self):
        pass

    def prioridad(self, r):
        sector = r.get("sector")

        if sector == "salud":
            if r.get("edad", 0) >= 65 or r.get("sintomas") == "severo":
                return "alta"
            return "baja"

        if sector == "retail":
            if r.get("gasto", 0) >= 300 and r.get("visitas", 0) >= 3:
                return "alta"
            return "baja"

        if sector == "banca":
            if r.get("score", 0) >= 700 and r.get("ratio", 1) <= 0.35:
                return "alta"
            return "baja"

        return "baja"

registros = [
    {"sector": "salud", "edad": 72, "sintomas": "leve"},
    {"sector": "salud", "edad": 30, "sintomas": "severo"},
    {"sector": "retail", "gasto": 320, "visitas": 4},
    {"sector": "retail", "gasto": 120, "visitas": 5},
    {"sector": "banca", "score": 740, "ratio": 0.33},
    {"sector": "banca", "score": 680, "ratio": 0.28},
]

engine = FeatureRuleEngine()

alta = 0
baja = 0
for r in registros:
    r["prioridad"] = engine.prioridad(r)
    if r["prioridad"] == "alta":
        alta += 1
    else:
        baja += 1

print("alta:", alta, "baja:", baja)
for r in registros:
    print(r)

# Resumen final (para mentor)

### Señales de comprensión
- El alumno puede explicar con sus palabras:
  - clase vs objeto
  - qué es un atributo
  - qué es un método
  - qué hace `__init__`
  - qué significa `self`
  - diferencia entre atributo de clase e instancia
  - composición vs herencia

### Señales de nivel aplicado (DS)
- El alumno puede crear una clase para:
  - limpiar datos
  - calcular métricas
  - encapsular reglas
  - evaluar predicciones
